# NVFP4 model serving with vLLM 0.28.0 on a Turing T4

This notebook is a proof of concept for running NVFP4 model checkpoints on a T4 with vLLM. To limit runtime, it uses the public vLLM 0.28.0 T4 wheelhouse and `nvidia/NVIDIA-Nemotron-Nano-9B-v2-NVFP4`. Because Turing has no BF16 support, activations use FP16.

The model download starts early in the background and logs explicit
`MODEL_DOWNLOAD_COMPLETE`, `MODEL_DOWNLOAD_FAILED`, or
`MODEL_DOWNLOAD_ABORTED` markers. Setup cells continue while the model
is downloading. vLLM startup is not wall-clock capped. Individual HTTP
probes are capped so a single `curl` cannot hang.

The model is defined by the `MODEL_*` variables in the next cell. To substitute another model, provide its Hugging Face repository path. The free Google Colab T4 runtime has 16 GB of nominal device memory (14.56 GiB visible), which must hold the model, activations, and kernels. Pick a larger device to run larger models.

## Trying other checkpoints

The model is set in one place, the environment block in the next cell, and any NVFP4 checkpoint
vLLM can load will work. Two alternatives are commented there: `nvidia/NVIDIA-Nemotron-3-Nano-4B-NVFP4`,
which is the same architecture family and quicker on a T4, and `mgoin/Qwen3-0.6B-NVFP4`, which is
smaller still and has no recurrent layers. The Qwen one is a compressed-tensors checkpoint and
declares its own quantization, so it needs `MODEL_QUANT=compressed-tensors`; the Nemotron
checkpoints carry no `quantization_config` and need `modelopt_mixed` named explicitly.

Nothing here is tuned for a particular checkpoint. The limits are the 14.56 GiB visible to vLLM
on the T4 and the free Colab session, not the format.


In [ ]:
%%bash
set -euxo pipefail

export DEBIAN_FRONTEND=noninteractive
export PIP_BREAK_SYSTEM_PACKAGES=1
export PYTHONNOUSERSITE=1
export PYTHONUNBUFFERED=1

cat > /content/nvfp4_env.sh <<'EOF'
export DEBIAN_FRONTEND=noninteractive
export PIP_BREAK_SYSTEM_PACKAGES=1
export PYTHONNOUSERSITE=1
export PYTHONUNBUFFERED=1
export PYTHONFAULTHANDLER=1
export HF_HUB_ENABLE_HF_TRANSFER=1
export CUDA_HOME=/usr/local/cuda
export PATH="${CUDA_HOME}/bin:${PATH}"
# Any NVFP4 checkpoint vLLM can load will do. Set these four together and rerun
# from this cell. MODEL_QUANT is needed because a checkpoint may or may not
# declare its own quantization: the Nemotron ones carry no quantization_config,
# so it has to be named, while compressed-tensors checkpoints declare theirs.
export MODEL_REPO=nvidia/NVIDIA-Nemotron-Nano-9B-v2-NVFP4
export MODEL_NAME=nemotron-nano-9b-v2-nvfp4-sanity
export MODEL_DIR=/content/models/nemotron-nano-9b-v2-nvfp4
export MODEL_QUANT=modelopt_mixed
export MODEL_OPTIONS=""
# export MODEL_OPTIONS="--attention-backend TRITON_ATTN --linear-backend marlin"

# Smaller and quicker on a T4, same architecture family:
# export MODEL_REPO=nvidia/NVIDIA-Nemotron-3-Nano-4B-NVFP4
# export MODEL_NAME=nemotron-3-nano-4b-nvfp4-sanity
# export MODEL_DIR=/content/models/nemotron-3-nano-4b-nvfp4
# export MODEL_QUANT=modelopt_mixed

# Smallest; it has no recurrent layers:
# export MODEL_REPO=mgoin/Qwen3-0.6B-NVFP4
# export MODEL_NAME=qwen3-0.6b-nvfp4-sanity
# export MODEL_DIR=/content/models/qwen3-0.6b-nvfp4
# export MODEL_QUANT=compressed-tensors
export WHEELHOUSE_ARCHIVE=/content/nvfp4_t4_wheelhouse.tgz
export WHEELHOUSE_DIR=/content/nvfp4_t4_wheelhouse
export WHEEL_DIR=/content/nvfp4_t4_wheelhouse/wheels
export LOG_DIR=/content/nvfp4_logs
export MODEL_DOWNLOAD_LOG=/content/nvfp4_logs/model_download.log
export SERVER_LOG=/content/nvfp4_logs/vllm_server.log
export REQUEST_LOG=/content/nvfp4_logs/request.log
EOF

. /content/nvfp4_env.sh
mkdir -p "${LOG_DIR}"

printf 'MODEL_REPO=%s\n' "${MODEL_REPO}"
printf 'MODEL_NAME=%s\n' "${MODEL_NAME}"
printf 'MODEL_DIR=%s\n' "${MODEL_DIR}"
printf 'MODEL_OPTIONS=%s\n' "${MODEL_OPTIONS}"
printf 'LOG_DIR=%s\n' "${LOG_DIR}"
python3 --version
free -h
df -h /content
nvidia-smi || true


Start the model download in the background.

In [ ]:
%%bash
set -euxo pipefail
. /content/nvfp4_env.sh
mkdir -p "${LOG_DIR}"
mkdir -p "$(dirname "${MODEL_DIR}")"

python3 -m pip install --upgrade pip
python3 -m pip install \
  huggingface_hub \
  hf_transfer

pid_file=/content/nvfp4_model_download.pid
if grep -q MODEL_DOWNLOAD_COMPLETE "${MODEL_DOWNLOAD_LOG}" \
  2>/dev/null; then
  echo "model download already complete"
  tail -80 "${MODEL_DOWNLOAD_LOG}" || true
  exit 0
fi

if [ -f "${pid_file}" ]; then
  old_pid="$(cat "${pid_file}")"
  if ps -p "${old_pid}" >/dev/null 2>&1; then
    echo "model download already running: ${old_pid}"
    tail -80 "${MODEL_DOWNLOAD_LOG}" || true
    exit 0
  fi
fi

: > "${MODEL_DOWNLOAD_LOG}"

(
  set -uxo pipefail
  . /content/nvfp4_env.sh
  trap 'date; echo MODEL_DOWNLOAD_ABORTED; exit 130' HUP INT TERM
  date
  hf download \
    "${MODEL_REPO}" \
    --repo-type model \
    --local-dir "${MODEL_DIR}"
  rc=$?
  date
  echo "hf download exit code: ${rc}"
  if [ "${rc}" = 0 ]; then
    echo MODEL_DOWNLOAD_COMPLETE
  else
    echo MODEL_DOWNLOAD_FAILED
  fi
  exit "${rc}"
) > "${MODEL_DOWNLOAD_LOG}" 2>&1 &

download_pid=$!
echo "${download_pid}" > "${pid_file}"
echo "download_pid=${download_pid}"
echo "model download log: ${MODEL_DOWNLOAD_LOG}"
sleep 2
ps -o pid,ppid,stat,etime,%mem,%cpu,cmd \
  -p "${download_pid}" || true
tail -80 "${MODEL_DOWNLOAD_LOG}" || true


Install system libraries.

In [ ]:
%%bash
set -euxo pipefail
. /content/nvfp4_env.sh

apt-get update
apt_pkgs=(
  ca-certificates
  curl
  git
  libxcb1
  python3-dev
  python3.12-dev
)
apt-get install -y --no-install-recommends "${apt_pkgs[@]}"

free -h
df -h /content
tail -40 "${MODEL_DOWNLOAD_LOG}" || true


Install PyTorch, Transformers, ModelOpt, and related packages.

In [ ]:
%%bash
set -euxo pipefail
. /content/nvfp4_env.sh

python3 -m pip install 'setuptools<82' wheel

need_torch=1
if python3 - <<'PY'
import sys
try:
    import torch
    import torchvision
    import torchaudio
except Exception:
    raise SystemExit(1)

ok = (
    torch.__version__ == "2.13.0+cu130"
    and torchvision.__version__ == "0.28.0+cu130"
    and torchaudio.__version__ == "2.11.0+cu130"
)
raise SystemExit(0 if ok else 1)
PY
then
  need_torch=0
fi

if [ "${need_torch}" = 1 ]; then
  torch_index=https://download.pytorch.org/whl/cu130
  python3 -m pip install \
    'torch==2.13.0+cu130' \
    'torchvision==0.28.0+cu130' \
    'torchaudio==2.11.0+cu130' \
    --index-url "${torch_index}"
else
  echo "torch packages already match cu130 targets"
fi

python3 -m pip install \
  'numpy<2.4' \
  safetensors \
  tqdm \
  'mistral_common>=1.10.0' \
  'transformers==5.8.0' \
  'nvidia-modelopt==0.43.0'

python3 - <<'PY'
import torch
print("torch", torch.__version__)
print("torch cuda", torch.version.cuda)
print("cuda available", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device", torch.cuda.get_device_name(0))
    print("capability", torch.cuda.get_device_capability(0))
PY

free -h
df -h /content
tail -40 "${MODEL_DOWNLOAD_LOG}" || true


Save the CUDA library paths to the environment file.

In [ ]:
%%bash
set -euxo pipefail
. /content/nvfp4_env.sh

python3 - <<'PY' > /content/nvfp4_cuda_libs.sh
from pathlib import Path
import site

dirs = []
for root in site.getsitepackages():
    base = Path(root)
    torch_lib = base / "torch" / "lib"
    if torch_lib.is_dir():
        dirs.append(str(torch_lib))
    nvidia_root = base / "nvidia"
    if nvidia_root.is_dir():
        for lib_dir in sorted(nvidia_root.glob("*/lib")):
            dirs.append(str(lib_dir))

lib_path = ":".join(dict.fromkeys(dirs))
print(f"export NVFP4_LIB_PATH={lib_path!r}")
print('export LD_LIBRARY_PATH="${NVFP4_LIB_PATH}:${LD_LIBRARY_PATH:-}"')
PY

cat /content/nvfp4_cuda_libs.sh
cat /content/nvfp4_cuda_libs.sh >> /content/nvfp4_env.sh
. /content/nvfp4_env.sh

python3 - <<'PY'
import torch
print("torch", torch.__version__)
print("torch cuda", torch.version.cuda)
PY


Download and unpack the prebuilt vLLM wheelhouse. Building locally exceeds the free Google Colab T4 allocation.

In [ ]:
%%bash
set -euxo pipefail
. /content/nvfp4_env.sh

WHEELHOUSE_BASE=https://huggingface.co/datasets/mgschwind
WHEELHOUSE_REPO=fleetwide-nvfp4-t4-wheelhouse
WHEELHOUSE_KIND=resolve/main
WHEELHOUSE_FILE=nvfp4_t4_vllm_0.28.0_cu130_py312_wheelhouse.tgz
WHEELHOUSE_URL="${WHEELHOUSE_BASE}/${WHEELHOUSE_REPO}"
WHEELHOUSE_URL="${WHEELHOUSE_URL}/${WHEELHOUSE_KIND}"
WHEELHOUSE_URL="${WHEELHOUSE_URL}/${WHEELHOUSE_FILE}"

curl   -L   --fail   --retry 5   --retry-delay 5   --connect-timeout 10   --max-time 900   "${WHEELHOUSE_URL}"   -o "${WHEELHOUSE_ARCHIVE}"

rm -rf "${WHEELHOUSE_DIR}"
mkdir -p "${WHEELHOUSE_DIR}"
tar -xzf "${WHEELHOUSE_ARCHIVE}" -C "${WHEELHOUSE_DIR}"
find "${WHEEL_DIR}" -maxdepth 1 -type f -name '*.whl' -print

du -sh "${WHEELHOUSE_DIR}"
free -h
df -h /content
tail -40 "${MODEL_DOWNLOAD_LOG}" || true


Install and verify the prebuilt vLLM wheel.

In [ ]:
%%bash
set -euxo pipefail
. /content/nvfp4_env.sh

vllm_wheel="$(find "${WHEEL_DIR}" -name 'vllm*.whl' -print -quit)"
need_vllm=1
if python3 - <<'PY'
try:
    import vllm
except Exception:
    raise SystemExit(1)
raise SystemExit(0 if vllm.__version__ == "0.28.0" else 1)
PY
then
  need_vllm=0
fi

if [ "${need_vllm}" = 1 ]; then
  python3 -m pip install "${vllm_wheel}"
else
  echo "expected vLLM wheel already installed"
fi

python3 - <<'PY'
import torch
import vllm
print("torch", torch.__version__)
print("torch cuda", torch.version.cuda)
print("vllm", vllm.__version__)
PY

free -h
df -h /content
tail -40 "${MODEL_DOWNLOAD_LOG}" || true


Confirm the model download has finished.

In [ ]:
%%bash
set -euxo pipefail
. /content/nvfp4_env.sh

pid_file=/content/nvfp4_model_download.pid
if [ ! -f "${pid_file}" ]; then
  echo "missing download pid file: ${pid_file}"
  exit 1
fi

download_pid="$(cat "${pid_file}")"
echo "download_pid=${download_pid}"
echo "model download log: ${MODEL_DOWNLOAD_LOG}"

if grep -q MODEL_DOWNLOAD_COMPLETE "${MODEL_DOWNLOAD_LOG}"; then
  tail -120 "${MODEL_DOWNLOAD_LOG}" || true
else
  tail -n +1 --pid="${download_pid}" -F "${MODEL_DOWNLOAD_LOG}" |
    sed \
      -e '/MODEL_DOWNLOAD_COMPLETE/q' \
      -e '/MODEL_DOWNLOAD_FAILED/q' \
      -e '/MODEL_DOWNLOAD_ABORTED/q'
fi

if grep -q MODEL_DOWNLOAD_COMPLETE "${MODEL_DOWNLOAD_LOG}"; then
  echo MODEL_DOWNLOAD_COMPLETE
else
  echo "model download did not complete successfully"
  exit 1
fi

du -sh "${MODEL_DIR}"
find "${MODEL_DIR}" -maxdepth 1 -type f -print
free -h
df -h /content
nvidia-smi || true


## Optional offline diagnostic

Run this cell to verify model loading and one generation without HTTP serving. Skip it for the fastest path to `vllm serve`. It still has to initialize vLLM and may take a long time on a T4.

As a sanity check, we call `llm.generate` with a simple question: `llm.generate(["What is the capital of Austria? Answer with exactly one word."], params)`. The correct answer, `Vienna`, appears at the end of the diagnostic output.

In [ ]:
%%bash
set -euxo pipefail
. /content/nvfp4_env.sh

LLM_OBJECT_LOG=/content/nvfp4_logs/llm_object.log
: > "${LLM_OBJECT_LOG}"

export CUDA_VISIBLE_DEVICES=0
export VLLM_LOGGING_LEVEL=DEBUG
export VLLM_LOG_STATS_INTERVAL=1
export PYTHONFAULTHANDLER=1
export TORCH_SHOW_CPP_STACKTRACES=1
export NCCL_DEBUG=INFO
export VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS=0

python3 -u - <<'PY' 2>&1 | tee -a "${LLM_OBJECT_LOG}"
import faulthandler
import os
import sys
import time

faulthandler.enable()
faulthandler.dump_traceback_later(120, repeat=True, file=sys.stderr)

print("STEP import vllm begin", flush=True)
from vllm import LLM, SamplingParams
print("STEP import vllm done", flush=True)

model = "/content/models/nemotron-nano-9b-v2-nvfp4"

print("STEP construct LLM begin", flush=True)
t0 = time.time()
llm = LLM(
    model=model,
    tokenizer=model,
    quantization=os.environ.get("MODEL_QUANT", "modelopt_mixed"),
    tensor_parallel_size=1,
    trust_remote_code=True,
    dtype="float16",
    max_model_len=512,
    max_num_batched_tokens=512,
    max_num_seqs=1,
    gpu_memory_utilization=0.88,
    cpu_offload_gb=0,
)
print("STEP construct LLM done", round(time.time() - t0, 2), flush=True)

params = SamplingParams(max_tokens=8, temperature=0)
print("STEP generate begin", flush=True)
t0 = time.time()
out = llm.generate(["What is the capital of Austria? Answer with exactly one word."], params)
print("STEP generate done", round(time.time() - t0, 2), flush=True)
print(out[0].outputs[0].text, flush=True)
PY

printf 'offline LLM log: %s\n' "${LLM_OBJECT_LOG}"


## Optional `vLLM serve` test

Run the following two cells to test model loading and one generation through the vLLM HTTP server. This test initializes both vLLM and the HTTP service and will take even longer than the offline diagnostic.


In [ ]:
%%bash
set -euxo pipefail
. /content/nvfp4_env.sh

export CUDA_VISIBLE_DEVICES=0
export VLLM_LOGGING_LEVEL=DEBUG
export VLLM_LOG_STATS_INTERVAL=1
export PYTHONFAULTHANDLER=1
export TORCH_SHOW_CPP_STACKTRACES=1
export NCCL_DEBUG=INFO
export VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS=0

: > "${SERVER_LOG}"
free -h
df -h /content
nvidia-smi || true

echo "vLLM log: ${SERVER_LOG}"
tail -n +1 -F "${SERVER_LOG}" |
  sed '/NVFP4_HEALTH_READY/q' &
log_tail_pid=$!

vllm serve "${MODEL_DIR}" \
  --served-model-name "${MODEL_NAME}" \
  --tokenizer "${MODEL_DIR}" \
  --quantization "${MODEL_QUANT}" \
  --tensor-parallel-size 1 \
  --trust-remote-code \
  --dtype float16 \
  --max-model-len 512 \
  --max-num-batched-tokens 512 \
  --max-num-seqs 1 \
  --gpu-memory-utilization 0.88 \
  --cpu-offload-gb 0 \
  ${MODEL_OPTIONS} \
  --uvicorn-log-level debug \
  --enable-log-requests \
  --enable-log-outputs \
  --log-error-stack \
  --host 127.0.0.1 \
  --port 8000 \
  > "${SERVER_LOG}" 2>&1 &

server_pid=$!
echo "${server_pid}" > /content/nvfp4_vllm_server.pid
echo "server_pid=${server_pid}"

ready=0
poll=1
while [ "${ready}" = 0 ]; do
  echo "health poll ${poll}"
  date
  ps -o pid,ppid,stat,etime,%mem,%cpu,cmd \
    -p "${server_pid}" || true
  free -h
  nvidia-smi || true

  if curl \
    --fail-with-body \
    --connect-timeout 2 \
    --max-time 5 \
    http://127.0.0.1:8000/health; then
    ready=1
    break
  fi

  if ! ps -p "${server_pid}" >/dev/null 2>&1; then
    echo "vLLM server exited before health became ready"
    wait "${server_pid}"
  fi

  poll=$((poll + 1))
  sleep 10
done

echo NVFP4_HEALTH_READY >> "${SERVER_LOG}"
wait "${log_tail_pid}" || true

echo "server ready on http://127.0.0.1:8000"
echo "vLLM log: ${SERVER_LOG}"
tail -200 "${SERVER_LOG}" || true


Use `curl` to run a model sanity check:

In [ ]:
%%bash
set -uxo pipefail
. /content/nvfp4_env.sh

: > "${REQUEST_LOG}"

curl \
  --fail-with-body \
  --connect-timeout 5 \
  --max-time 300 \
  -X POST \
  http://127.0.0.1:8000/v1/chat/completions \
  -H 'Content-Type: application/json' \
  --data-binary @- <<JSON 2>&1 | tee -a "${REQUEST_LOG}"
{
  "model": "${MODEL_NAME}",
  "messages": [
    {
      "role": "system",
      "content": "/no_think"
    },
    {
      "role": "user",
      "content": "What is the capital of Austria? Answer with exactly one word."
    }
  ],
  "max_tokens": 8,
  "temperature": 0
}
JSON
curl_rc=${PIPESTATUS[0]}

echo "curl_rc=${curl_rc}"
printf 'request log: %s\n' "${REQUEST_LOG}"
printf 'vLLM log: %s\n' "${SERVER_LOG}"
cat "${REQUEST_LOG}" || true
ps -o pid,ppid,stat,etime,%mem,%cpu,cmd \
  -p "$(cat /content/nvfp4_vllm_server.pid)" || true
nvidia-smi || true
echo "--- vLLM log tail 400 ---"
tail -400 "${SERVER_LOG}" || true
echo "--- end vLLM log tail 400 ---"
exit "${curl_rc}"
